# Image Denoising Autoencoder (PyTorch, from scratch)

Built with raw `nn.Module`, a manual training loop, and manual MSE — no `model.fit()`, no high level wrappers.

Dataset: FashionMNIST (28x28 grayscale).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)  # so results are reproducible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

## 1. Data

Just the raw tensors, scaled to [0, 1]. No normalization to mean/std since we want pixel values to stay in [0,1] for the noise + sigmoid output to make sense.

In [ ]:
transform = transforms.ToTensor()  # converts PIL image -> tensor in [0, 1]

train_data = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_data = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"train samples: {len(train_data)}, test samples: {len(test_data)}")

## 2. Noise function

We add Gaussian noise on the fly during training (not baked into the dataset), so the model sees a slightly different noisy version of each image every epoch.

In [ ]:
def add_gaussian_noise(images, noise_factor=0.3):
    # images: tensor in [0, 1], shape (B, 1, 28, 28)
    noise = torch.randn_like(images) * noise_factor
    noisy = images + noise
    noisy = torch.clamp(noisy, 0., 1.)  # keep pixel values valid after adding noise
    return noisy

## 3. Model

Encoder: Conv -> ReLU -> MaxPool, twice (downsamples 28x28 -> 7x7).

Decoder: ConvTranspose -> ReLU -> ConvTranspose -> Sigmoid (upsamples back to 28x28).

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 x 28 x 28 -> 16 x 28 x 28 -> pooled -> 16 x 14 x 14
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        # 16 x 14 x 14 -> 32 x 14 x 14 -> pooled -> 32 x 7 x 7
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        return x  # shape: (B, 32, 7, 7), this is our compressed "latent" representation


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 32 x 7 x 7 -> 16 x 14 x 14
        self.deconv1 = nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2, stride=2)
        # 16 x 14 x 14 -> 1 x 28 x 28
        self.deconv2 = nn.ConvTranspose2d(in_channels=16, out_channels=1, kernel_size=2, stride=2)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # squashes output back into [0, 1] pixel range

    def forward(self, x):
        x = self.relu(self.deconv1(x))
        x = self.sigmoid(self.deconv2(x))
        return x


class DenoisingAutoencoder(nn.Module):
    # just glues encoder + decoder together
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        latent = self.encoder(x)
        out = self.decoder(latent)
        return out

In [ ]:
model = DenoisingAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
noise_factor = 0.3

total_params = sum(p.numel() for p in model.parameters())
print(f"total trainable params: {total_params:,}")

## 4. Training loop

Fully manual: noisy input -> forward pass -> MSE against the *clean* image -> backward -> step. No `.fit()` anywhere.

In [ ]:
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, _ in train_loader:   # labels not needed, we only care about reconstruction
        images = images.to(device)
        noisy_images = add_gaussian_noise(images, noise_factor)

        optimizer.zero_grad()

        outputs = model(noisy_images)            # forward pass on the noisy input
        loss = torch.mean((outputs - images) ** 2)  # manual MSE: target is the CLEAN image

        loss.backward()     # backprop
        optimizer.step()    # update weights

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_data)
    train_losses.append(epoch_loss)
    print(f"epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.6f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("training loss")
plt.grid(True)
plt.show()

## 5. Results: noisy input vs denoised output

Grab one batch from the test set, add noise, run it through the trained model, and compare.

In [ ]:
model.eval()

images, _ = next(iter(test_loader))
images = images.to(device)
noisy_images = add_gaussian_noise(images, noise_factor)

with torch.no_grad():   # no need to track gradients just for viewing results
    reconstructed = model(noisy_images)

# move everything back to cpu + numpy for plotting
noisy_images = noisy_images.cpu().numpy()
reconstructed = reconstructed.cpu().numpy()

n = 8  # how many examples to show
fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))

for i in range(n):
    axes[0, i].imshow(noisy_images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("noisy", loc="left")

    axes[1, i].imshow(reconstructed[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("denoised", loc="left")

plt.tight_layout()
plt.show()